In [1]:
import os

print("--- Zawartość katalogu wejściowego ---")
for root, dirs, files in os.walk('/kaggle/input/'):
    # Pokazuje tylko strukturę katalogów do 3 poziomów głębokości, żeby nie zaśmiecać ekranu plikami
    level = root.replace('/kaggle/input/', '').count(os.sep)
    if level < 3:
        indent = ' ' * 4 * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        for d in dirs:
            print(f"{indent}    ├── 📂 {d}")
        break # Przerywamy po pierwszym poziomie, żeby zobaczyć główną nazwę folderu
!ls /kaggle/input/datasets
!ls /kaggle/input/datasets/tristanzhang32
!ls /kaggle/input/datasets/tristanzhang32/ai-generated-images-vs-real-images


import shutil
import os

folder_do_usuniecia = '/kaggle/working/dataset_pociety'

if os.path.exists(folder_do_usuniecia):
    print(f"🔄 Folder {folder_do_usuniecia} istnieje. Rozpoczynam czyszczenie...")
    shutil.rmtree(folder_do_usuniecia)
    print("✅ Folder został całkowicie usunięty!")
else:
    print("ℹ️ Folder dataset_pociety nie istnieje w /kaggle/working/ – dysk jest czysty.")

--- Zawartość katalogu wejściowego ---
📁 /
    ├── 📂 datasets
tristanzhang32
ai-generated-images-vs-real-images
test  train
ℹ️ Folder dataset_pociety nie istnieje w /kaggle/working/ – dysk jest czysty.


In [2]:
import os
import cv2
import glob
from tqdm import tqdm

def slice_dataset_balanced(source_dir, target_dir, tile_size=224, stride=224, max_images_per_class=1250):
    # Automatycznie wykrywamy podfoldery klas (np. fake i real)
    classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    print(f"📁 Wykryte klasy w {os.path.basename(source_dir)}: {classes}")
    
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    
    for cls in classes:
        cls_source_dir = os.path.join(source_dir, cls)
        image_paths = []
        for ext in extensions:
            image_paths.extend(glob.glob(os.path.join(cls_source_dir, '**', ext), recursive=True))
            
        print(f"  ↳ Klasa '{cls}': znaleziono łącznie {len(image_paths)} dużych zdjęć.")
        
        # Bezpieczne ograniczenie osobno dla każdej klasy!
        if len(image_paths) > max_images_per_class:
            print(f"  ⚠️ Ograniczam klasę '{cls}' do {max_images_per_class} zdjęć dla równowagi dyskowej.")
            image_paths = image_paths[:max_images_per_class]
            
        print(f"  🪓 Cięcie klasy '{cls}'...")
        for path in tqdm(image_paths, desc=f"Klasa {cls}"):
            img = cv2.imread(path)
            if img is None:
                continue
                
            h, w, _ = img.shape
            base_name = os.path.splitext(os.path.basename(path))[0]
            
            # Zapisujemy bezpośrednio do target_dir/klasa/
            out_folder = os.path.join(target_dir, cls)
            os.makedirs(out_folder, exist_ok=True)
            
            tile_count = 0
            for y in range(0, h - tile_size + 1, stride):
                for x in range(0, w - tile_size + 1, stride):
                    tile = img[y:y+tile_size, x:x+tile_size]
                    if tile.shape[0] != tile_size or tile.shape[1] != tile_size:
                        continue
                    
                    tile_name = f"{base_name}_tile_{tile_count}.jpg"
                    cv2.imwrite(os.path.join(out_folder, tile_name), tile)
                    tile_count += 1


SOURCE_TRAIN = '/kaggle/input/datasets/tristanzhang32/ai-generated-images-vs-real-images/train'
SOURCE_VAL = '/kaggle/input/datasets/tristanzhang32/ai-generated-images-vs-real-images/test'

TARGET_TRAIN = '/kaggle/working/dataset_pociety/train'
TARGET_VAL = '/kaggle/working/dataset_pociety/test'

# Czyścimy stary, błędny folder roboczy, żeby nie mieszać danych
import shutil
if os.path.exists('/kaggle/working/dataset_pociety'):
    shutil.rmtree('/kaggle/working/dataset_pociety')

print("\n--- Zbalansowane Przygotowanie Kafelków 1:1 ---")
# Trening: po 1250 zdjęć z fake i real (łącznie 2500 dużych zdjęć -> masa kafelków)
slice_dataset_balanced(SOURCE_TRAIN, TARGET_TRAIN, tile_size=224, stride=224, max_images_per_class=1250)
# Test: po 300 zdjęć z fake i real (łącznie 600 dużych zdjęć)
slice_dataset_balanced(SOURCE_VAL, TARGET_VAL, tile_size=224, stride=224, max_images_per_class=300)


--- Zbalansowane Przygotowanie Kafelków 1:1 ---
📁 Wykryte klasy w train: ['fake', 'real']
  ↳ Klasa 'fake': znaleziono łącznie 24000 dużych zdjęć.
  ⚠️ Ograniczam klasę 'fake' do 1250 zdjęć dla równowagi dyskowej.
  🪓 Cięcie klasy 'fake'...


Klasa fake: 100%|██████████| 1250/1250 [00:39<00:00, 31.71it/s]


  ↳ Klasa 'real': znaleziono łącznie 24000 dużych zdjęć.
  ⚠️ Ograniczam klasę 'real' do 1250 zdjęć dla równowagi dyskowej.
  🪓 Cięcie klasy 'real'...


Klasa real: 100%|██████████| 1250/1250 [03:51<00:00,  5.40it/s]


📁 Wykryte klasy w test: ['fake', 'real']
  ↳ Klasa 'fake': znaleziono łącznie 6000 dużych zdjęć.
  ⚠️ Ograniczam klasę 'fake' do 300 zdjęć dla równowagi dyskowej.
  🪓 Cięcie klasy 'fake'...


Klasa fake: 100%|██████████| 300/300 [00:10<00:00, 27.72it/s]


  ↳ Klasa 'real': znaleziono łącznie 6000 dużych zdjęć.
  ⚠️ Ograniczam klasę 'real' do 300 zdjęć dla równowagi dyskowej.
  🪓 Cięcie klasy 'real'...


Klasa real: 100%|██████████| 300/300 [00:59<00:00,  5.08it/s]


In [3]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

#base_dir = 'deepfake_faces_data/Final Dataset'
train_dir = '/kaggle/working/dataset_pociety/train'
test_dir = '/kaggle/working/dataset_pociety/test'
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

print("\n--- ładowanie zbioru treningowego ---")
train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("\n--- ładowanie zbioru testowego ---")
val_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("\n--- Weryfikacja matematyczna klas przez Keras ---")
for i, name in enumerate(class_names):
    print(f"Wartość {i}.0 na wyjściu modelu (Sigmoid) oznacza klasę: {name}")

2026-05-24 09:35:08.115578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779615308.303942      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779615308.353466      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779615308.796248      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779615308.796288      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779615308.796292      57 computation_placer.cc:177] computation placer alr


--- ładowanie zbioru treningowego ---
Found 218874 files belonging to 2 classes.


I0000 00:00:1779615337.442913      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



--- ładowanie zbioru testowego ---
Found 54912 files belonging to 2 classes.

--- Weryfikacja matematyczna klas przez Keras ---
Wartość 0.0 na wyjściu modelu (Sigmoid) oznacza klasę: fake
Wartość 1.0 na wyjściu modelu (Sigmoid) oznacza klasę: real


In [4]:
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = "imagenet"
)

base_model.trainable = False



model = models.Sequential([
    layers.RandomFlip("horizontal", input_shape=(224, 224, 3)),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [5]:
print("--start training--")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

--start training--
Epoch 1/5


I0000 00:00:1779615346.839205     140 service.cc:152] XLA service 0x7fcb50002560 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779615346.839254     140 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1779615347.938987     140 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779615355.193946     140 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6840/6840 ━━━━━━━━━━━━━━━━━━━━ 203s 28ms/step - accuracy: 0.8906 - loss: 0.2783 - val_accuracy: 0.8809 - val_loss: 0.2809
Epoch 2/5
6840/6840 ━━━━━━━━━━━━━━━━━━━━ 177s 26ms/step - accuracy: 0.9028 - loss: 0.2500 - val_accuracy: 0.8826 - val_loss: 0.2785
Epoch 3/5
6840/6840 ━━━━━━━━━━━━━━━━━━━━ 179s 26ms/step - accuracy: 0.9034 - loss: 0.2496 - val_accuracy: 0.8817 - val_loss: 0.2807
Epoch 4/5
6840/6840 ━━━━━━━━━━━━━━━━━━━━ 177s 26ms/step - accuracy: 0.9036 - loss: 0.2491 - val_accuracy: 0.8781 - val_loss: 0.2859
Epoch 5/5
6840/6840 ━━━━━━━━━━━━━━━━━━━━ 175s 26ms/step - accuracy: 0.9022 - loss: 0.2499 - val_accuracy: 0.8834 - val_loss: 0.2788


In [ ]:
print("--Dostrajanie modelu--")

unfreezeModel = model.layers[4]
unfreezeModel.trainable = True

for layer in unfreezeModel.layers[:-30]:
  layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history_f = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=7
)


--Dostrajanie modelu--


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,527,681 (5.83 MB)

...

Epoch 1/7


2026-05-24 09:51:03.754103: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-24 09:51:03.951683: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


6837/6840 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8924 - loss: 0.2656

2026-05-24 09:54:17.283073: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-24 09:54:17.482072: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


6840/6840 ━━━━━━━━━━━━━━━━━━━━ 249s 34ms/step - accuracy: 0.8924 - loss: 0.2656 - val_accuracy: 0.9077 - val_loss: 0.2269
Epoch 2/7
6840/6840 ━━━━━━━━━━━━━━━━━━━━ 219s 32ms/step - accuracy: 0.9337 - loss: 0.1709 - val_accuracy: 0.9100 - val_loss: 0.2255
Epoch 3/7
6839/6840 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9445 - loss: 0.1450

In [ ]:
# Zapisanie wyszkolonego modelu V2
model_save_path = '/kaggle/working/deepfake_v2.keras'
model.save(model_save_path)

print(f"Model został pomyślnie zapisany w: {model_save_path}")

In [ ]:
# !pip install -q gradio

# import gradio as gr
# import tensorflow as tf
# import numpy as np

# # 1. Ładowanie zapisanego przed chwilą modelu
# print("🔄 Ładowanie modelu z dysku Kaggle...")
# loaded_model = tf.keras.models.load_model(
#     '/kaggle/working/deepfake_v2.keras',
#     custom_objects={'preprocess_input': tf.keras.applications.mobilenet_v2.preprocess_input}
# )

# def predict_deepfake_scan(img):
#     if img is None:
#         return "Proszę wrzucić zdjęcie."

#     # Skala 1:1, brak zgniatania do 600 px
#     img = np.array(img, dtype=np.uint8)
#     img_float = img.astype(np.float32)

#     # Globalny preprocessing (statystyki RGB)
#     img_ready = np.expand_dims(img_float, axis=0)
#     img_ready = tf.keras.applications.mobilenet_v2.preprocess_input(img_ready)
#     img_ready = img_ready[0].numpy() if hasattr(img_ready, 'numpy') else img_ready[0]

#     h, w, _ = img_ready.shape
#     tile_size = 224
#     stride = 224  

#     all_scores = []

#     # Kafelkowanie
#     for y in range(0, h - tile_size + 1, stride):
#         for x in range(0, w - tile_size + 1, stride):
#             tile = img_ready[y:y+tile_size, x:x+tile_size]
#             if tile.shape[0] != tile_size or tile.shape[1] != tile_size:
#                 continue
            
#             tile_array = np.expand_dims(tile, axis=0)
#             prediction = loaded_model.predict(tile_array, verbose=0)
#             all_scores.append(prediction[0][0])

#     if not all_scores:
#         return "Zdjęcie jest za małe na analizę kafelkową (minimum 224x224 px)."

#     final_score = np.mean(all_scores)

#     if final_score < 0.5:
#         confidence = (1 - final_score) * 100
#         return f"WYKRYTO GENERACJĘ AI! (Pewność ogólna: {confidence:.2f}%)"
#     else:
#         confidence = final_score * 100
#         return f"PRAWDZIWE ZDJĘCIE (Pewność ogólna: {confidence:.2f}%)"

# # 3. Uruchomienie interfejsu
# interface = gr.Interface(
#     fn=predict_deepfake_scan,
#     inputs=gr.Image(),
#     outputs="text",
#     title="Wykrywacz Deepfake V2 (Skala 1:1)",
#     description="Prototyp działający na infrastrukturze Kaggle przy użyciu kafelkowania."
# )

# # Flaga share=True jest kluczowa, wygeneruje publiczny link gradio.live
# interface.launch(share=True)